In [92]:
from typing import Optional

import conllu
import pandas as pd
from conllu import parse_tree

In [229]:
sentences = []
with open("train_nlprepl-ud.conllu", "r", encoding="utf-8") as f:
    for sentence in conllu.parse_incr(f):
        sentences.append(sentence)

In [240]:
# Load CSV and set 'lemma' as string index
morph_dict = pd.read_csv(
    "dictionary.v3.csv",
    dtype={"lemma": "string"}  # ensure lemma is treated as string
)
morph_dict.set_index("lemma", inplace=True)

print("Supported tags", morph_dict.columns)

def has_lemma(lemma: str):
    return lemma in morph_dict.index

poliform_tag_by_ud_tag = {
    "ppron3:sg:nom:m3:ter:akc:npraep": "ppron3:sg:nom:m1.m2.m3:ter:_:_",
    "praet:sg:m1:perf": "praet:sg:m1.m2.m3:perf",
    "praet:sg:m2:perf": "praet:sg:m1.m2.m3:perf",
    "praet:sg:m3:perf": "praet:sg:m1.m2.m3:perf",
    "praet:sg:m1:imperf": "praet:sg:m1.m2.m3:imperf",
    "praet:sg:m2:imperf": "praet:sg:m1.m2.m3:imperf",
    "praet:sg:m3:imperf": "praet:sg:m1.m2.m3:imperf",
}

def get_form(lemma:str, tag: str) -> str:
    if not has_lemma(lemma):
        return None
    row = morph_dict.loc[lemma]

    if tag in poliform_tag_by_ud_tag:
        tag = poliform_tag_by_ud_tag[tag]

    if tag not in row.index:
        print("Unsupported tag", tag)
        return None

    val = row[tag]
    if pd.notna(val) and val != "":
        return val
    else:
        return None

Supported tags Index(['fin:sg:pri:perf', 'fin:sg:pri:imperf', 'fin:sg:sec:perf',
       'fin:sg:sec:imperf', 'fin:sg:ter:perf', 'fin:sg:ter:imperf',
       'fin:pl:pri:perf', 'fin:pl:pri:imperf', 'fin:pl:sec:perf',
       'fin:pl:sec:imperf', 'fin:pl:ter:perf', 'fin:pl:ter:imperf',
       'praet:sg:f:perf', 'praet:sg:m1.m2.m3:perf', 'praet:sg:f:imperf',
       'praet:sg:m1.m2.m3:imperf'],
      dtype='object')


Predicate - token that:
* is root, or
* is the HEAD of a subject token (can be checked with 'deprel' dependency relation label)

In [ ]:
from copy import deepcopy
from tqdm import tqdm

def sentence_text(sent):
    """Conlu library doesn't have 'generate raw text' function"""
    out = []
    for tok in sent:
        if not isinstance(tok["id"], int):
            continue

        out.append(tok["form"])

        misc = tok.get("misc")
        if not misc or misc.get("SpaceAfter") != "No":
            out.append(" ")

    return "".join(out).rstrip()

In [39]:
s = sentences[2896]
print(s.metadata['text'])
tree = s.to_tree()
tree.print_tree()

In [230]:
s = sentences[54104]
tree = s.to_tree()
tree.print_tree()

(deprel:root) form:robisz lemma:robić upos:VERB [5]
    (deprel:advmod) form:czemu lemma:czemu upos:ADV [2]
        (deprel:advmod:emph) form:a lemma:a upos:CCONJ [1]
    (deprel:nsubj) form:ty lemma:ty upos:PRON [3]
    (deprel:advmod:neg) form:nie lemma:nie upos:PART [4]
    (deprel:punct) form:? lemma:? upos:PUNCT [6]


In [87]:
def match_predicate_with_pron_nsubj(sentence) -> Optional[conllu.Token]:
    # For now lets take only root predicates
    root = sentence.to_tree()
    token = root.token

    upos = token['upos'] # Universal Part-of-Speech
    if upos != 'VERB':
        return None

    has_subj = False
    for ch in root.children:
        ch_token = ch.token
        if ch_token['deprel'] == 'nsubj' and ch_token['upos'] == "PRON":
            has_subj = True

    if not has_subj:
        return None

    return token

def change_morph(token, source_tag, target_tag):
    xpos = token['xpos'] # language-specific part-of-speech tag
    if xpos != source_tag:
        return None

    lemma = token["lemma"]
    if not has_lemma(lemma):
        return None

    is_capitalized = token['form'][0].isupper()

    target_form = get_form(lemma, target_tag)
    if target_form is None:
        return None

    if is_capitalized:
        target_form = target_form[:1].upper() + target_form[1:]

    token['form'] = target_form
    token['xpos'] = target_tag

    return token

(deprel:root) form:robisz lemma:robić upos:VERB [5]
    (deprel:advmod) form:czemu lemma:czemu upos:ADV [2]
        (deprel:advmod:emph) form:a lemma:a upos:CCONJ [1]
    (deprel:nsubj) form:ty lemma:ty upos:PRON [3]
    (deprel:advmod:neg) form:nie lemma:nie upos:PART [4]
    (deprel:punct) form:? lemma:? upos:PUNCT [6]


In [231]:
def run_filter_transform(match_token_fn, transform_inplace_fn, limit=None):
        # INLINED 'subj' filter
    rows = []

    for ix, sentence in tqdm(enumerate(sentences)):
        if limit is not None:
            if len(rows) >= limit:
                break

        sentence = deepcopy(sentence)  # to not break original objects

        token = match_token_fn(sentence)
        if token is None:
            continue

        modified_token = transform_inplace_fn(token)
        if modified_token is None:
            continue

        correct_text = sentence.metadata['text']
        incorrect_text = sentence_text(sentence)
        rows.append((ix, correct_text, incorrect_text))

    return pd.DataFrame(rows, columns=["conllu_index", "correct", "incorrect"])

In [116]:
from functools import partial

dfs = []

for time in ["perf", "imperf"]:
    persons = ["pri", "sec", "ter"]
    for source_person in persons:
        for target_person in persons:
            if source_person == target_person:
                continue

            source_tag = f'fin:sg:{source_person}:{time}'
            target_tag = f'fin:sg:{target_person}:{time}'

            transformation = partial(change_morph, source_tag=source_tag, target_tag=target_tag)
            dfs.append(run_filter_transform(match_predicate_with_pron_nsubj, transformation))

combined = pd.concat(dfs)
combined = combined.sort_values('conllu_index')
combined.to_csv("output_old/person_changed.csv", index=False)

In [127]:
def preview_sentences(match_token_fn, limit=5):
    matched = 0
    for ix, sentence in tqdm(enumerate(sentences)):
        if limit is not None:
            if matched >= limit:
                break
        token = match_token_fn(sentence)
        if token is not None:
            print(f"Conllu index={ix}, sentence= {sentence.metadata['text']}")
            matched += 1

69360it [00:11, 6028.19it/s] 
69360it [00:10, 6489.16it/s] 
69360it [00:10, 6388.29it/s] 
69360it [00:10, 6313.03it/s] 
69360it [00:11, 6177.45it/s] 
69360it [00:11, 6123.96it/s] 
69360it [00:11, 6302.21it/s] 
69360it [00:10, 6535.62it/s] 
69360it [00:10, 6621.72it/s] 
69360it [00:10, 6640.69it/s] 
69360it [00:10, 6601.30it/s] 
69360it [00:10, 6589.97it/s] 


In [173]:
def match_pred_with_pron_nsubj(sentence):
    root = sentence.to_tree()
    token = root.token

    upos = token['upos'] # Universal Part-of-Speech
    if upos != 'VERB':
        return None

    has_subj = False
    for ch in root.children:
        ch_token = ch.token
        if ch_token['deprel'] == 'nsubj' and ch_token['upos'] == "PRON":
            has_subj = True

    if not has_subj:
        return None

    return token

preview_sentences(partial(match_pred_with_pron_nsubj), limit=5)

1276it [00:00, 69765.64it/s]

Conllu index=112, sentence= A oni na przemian śpiewali i śmiali się z uciechy.
Conllu index=165, sentence= Lecz oni milczeli również ogarnięci właściwą powszechności niechęcią do wyjawienia swej opinii, nie poznawszy wprzód opinii drugich.
Conllu index=197, sentence= Jeżeli, mój Boże, ona tak pyta, to znaczy, coś się w niej przesiliło, chce żyć.
Conllu index=218, sentence= I wszyscy wnet pojęli, że ten okręt oznaczał Polskę, a była tam następnie mowa o nieładzie, co wkradł się do załogi, o burzy i o napadzie korsarzy, i o tym, jak okręt zaczął tonąć.
Conllu index=221, sentence= Nie pamiętała ona już domu swego zamożnym.
Conllu index=235, sentence= Lecz przekonał on się już nieraz, że przedmioty, których komu odmówił, traciły potem dla niego całą wartość.
Conllu index=354, sentence= Pod wpływem tych niewinnych zajęć przedstawiała się ona łagodniej, lecz i bardziej mglisto.
Conllu index=469, sentence= Lecz równie srogo zwykł był on występować przy rannych raportach; areszt i chłosta dykt

In [182]:
s = sentences[69149]
s.to_tree().print_tree()
print(s.to_tree().token['xpos'])
print(s.to_tree().children[0].token['xpos'])

(deprel:root) form:usunęła lemma:usunąć upos:VERB [8]
    (deprel:nsubj) form:Ja lemma:ja upos:PRON [1]
    (deprel:advmod) form:nigdy lemma:nigdy upos:ADV [3]
        (deprel:advmod:emph) form:osobiście lemma:osobiście upos:ADV [2]
    (deprel:aux:cnd) form:by lemma:by upos:AUX [4]
    (deprel:aux:clitic) form:m lemma:być upos:AUX [5]
    (deprel:obj) form:dziecka lemma:dziecko upos:NOUN [6]
    (deprel:advmod:neg) form:nie lemma:nie upos:PART [7]
    (deprel:punct) form:. lemma:. upos:PUNCT [9]
praet:sg:f:perf
ppron12:sg:nom:f:pri


In [181]:
dfs = []

for time in ["perf", "imperf"]:
    gender = ["f", "m2", "m3"]
    for source_gender in gender:
        for target_gender in gender:
            if source_gender == target_gender:
                continue
            if source_gender.startswith("m") and target_gender.startswith("m"):
                continue

            source_tag = f'praet:sg:{source_gender}:{time}'
            target_tag = f'praet:sg:{target_gender}:{time}'

            transformation = partial(change_morph, source_tag=source_tag, target_tag=target_tag)
            dfs.append(run_filter_transform(match_pred_with_pron_nsubj, transformation))

combined = pd.concat(dfs)
combined = combined.sort_values('conllu_index')
combined.to_csv("output_old/gender_changed.csv", index=False)

69360it [00:10, 6566.20it/s] 
69360it [00:10, 6469.01it/s] 
69360it [00:10, 6575.09it/s] 
69360it [00:10, 6486.40it/s] 
69360it [00:10, 6428.67it/s] 
69360it [00:10, 6526.77it/s] 
69360it [00:10, 6629.77it/s] 
69360it [00:10, 6515.19it/s] 


Problem - niespojność polimorf z UD, np:

* w UD mamy 'praet:pl:m1:imperf' a w polimorf nie ma
* w UD mamy 'ppron3:sg:nom:m3:ter:akc:npraep', a w polimorf nie ma  ( 'ppron3:sg:nom:m1.m2.m3:ter:_:_' ?)  (ona/on)

Iteracja:

* 69149,Ja osobiście nigdy bym dziecka nie usunęła.,Ja osobiście nigdy by m dziecka nie usunął.


In [246]:
def match_subj_verb_number_clause(sentence):
    tokenTree = sentence.to_tree()
    token = tokenTree.token
    if token['upos'] == 'VERB' and token['deprel'] == 'root' and 'sg' in token['xpos']:
        for ch in tokenTree.children:
            if ch.token['upos'] == 'VERB' and ch.token['deprel'] == 'csubj' and 'inf' in ch.token['xpos'].split(":"):
                return token
    else:
        return None


def change_number(token):
    mappings = {
        "pl": "sg",
        "sg": "pl"
    }

    lemma = token["lemma"]
    if not has_lemma(lemma):
        return None

    xpos = token['xpos'].split(":")

    for from_tag, to_tag in mappings.items():
        if from_tag in xpos:
            tag_ix = xpos.index(from_tag)
            target_xpos = xpos[:tag_ix] + [to_tag] + xpos[tag_ix+1:]
            target_xpos = ":".join(target_xpos)

            target_form = get_form(lemma, target_xpos)
            if target_form is None:
                print("Missing morph dict for", lemma, target_xpos)
                return None

            token['form'] = target_form
            token['xpos'] = target_xpos

            return token

    return None

# preview_sentences(match_subj_verb_number_clause)
df = run_filter_transform(match_subj_verb_number_clause, change_number)
# df

1665it [00:00, 5461.23it/s]

Unsupported tag praet:pl:n:imperf
Missing morph dict for zdawać praet:pl:n:imperf


33203it [00:05, 6923.87it/s]

Unsupported tag praet:pl:n:perf
Missing morph dict for przyjść praet:pl:n:perf


60745it [00:09, 6412.61it/s] 


KeyboardInterrupt: 

In [242]:
def match_subj_verb_number_pp_attractor(sentence: conllu.TokenList):
    tokenTree = sentence.to_tree()
    token = tokenTree.token
    token_xpos = token['xpos'].split(":")

    for number in ['sg', 'pl']:
        if token['upos'] == 'VERB' and token['deprel'] == 'root' and number in token_xpos:
            for ch in tokenTree.children:
                ch_token_xpos = ch.token['xpos'].split(":")
                if ch.token['upos'] == 'NOUN' and ch.token['deprel'] == 'nsubj' and number in ch_token_xpos:
                    for attractor in ch.children:
                        attractor.token['upos'] = 'NOUN'
                        attractor_xpos = attractor.token['xpos'].split(":")

                        other_number = 'pl' if number == 'sg' else 'sg'
                        if other_number in attractor_xpos:
                            for prep in attractor.children:
                                if prep.token['upos'] == 'ADP' and prep.token['deprel'] == 'case':
                                    return token

subj_verb_number_pp_attractor_df = run_filter_transform(match_subj_verb_number_pp_attractor, change_number)
subj_verb_number_pp_attractor_df

2841it [00:00, 5483.52it/s]

Unsupported tag praet:pl:f:perf
Missing morph dict for trafić praet:pl:f:perf


4475it [00:00, 5254.86it/s]

Unsupported tag praet:pl:n:perf
Missing morph dict for osłabić praet:pl:n:perf
Unsupported tag praet:sg:n:imperf
Missing morph dict for być praet:sg:n:imperf
Unsupported tag praet:pl:m3:perf
Missing morph dict for okazać praet:pl:m3:perf
Unsupported tag praet:pl:m3:perf:nagl
Missing morph dict for wzrosnąć praet:pl:m3:perf:nagl


5582it [00:01, 5209.52it/s]

...

Unsupported tag praet:pl:m3:imperf
Missing morph dict for otaczać praet:pl:m3:imperf
Unsupported tag praet:pl:n:imperf
Missing morph dict for być praet:pl:n:imperf
Unsupported tag praet:pl:m3:perf
Missing morph dict for przestać praet:pl:m3:perf
Unsupported tag praet:pl:m3:perf
Missing morph dict for zamienić praet:pl:m3:perf


67255it [00:10, 6557.81it/s]

Unsupported tag praet:pl:f:perf
Missing morph dict for wystąpić praet:pl:f:perf
Unsupported tag praet:pl:m1:perf
Missing morph dict for trafić praet:pl:m1:perf
Unsupported tag praet:pl:m3:perf
Missing morph dict for rozwiać praet:pl:m3:perf
Unsupported tag praet:pl:f:imperf
Missing morph dict for odbywać praet:pl:f:imperf
Unsupported tag praet:sg:n:imperf
Missing morph dict for trwać praet:sg:n:imperf
Unsupported tag praet:pl:n:imperf
Missing morph dict for mieć praet:pl:n:imperf


69360it [00:10, 6314.56it/s]


,conllu_index,correct,incorrect
0,1377,Nie wszystkie tomiki Broniewskiego sprzed roku...,Nie wszystkie tomiki Broniewskiego sprzed roku...
1,2477,Krążki VideoCD oprócz ścieżki audio/wideo mogą...,Krążki VideoCD oprócz ścieżki audio/wideo może...
2,2671,Towarzysze z Moskwy przeprowadzali w Polsce sz...,Towarzysze z Moskwy przeprowadzał w Polsce szk...
3,3034,1 lutego 1982 r. - pojawiły się talony na benz...,1 lutego 1982 r. - pojawił się talony na benzy...
4,3054,Popyt na usługi headhunterskie wyraźnie rośnie...,Popyt na usługi headhunterskie wyraźnie rosną ...
...,...,...,...
322,68195,O swojej lojalności - mimo wyroku niezawisłego...,O swojej lojalności - mimo wyroku niezawisłego...
323,68213,Policjanci z Inspektoratu Komendy Głównej usta...,Policjanci z Inspektoratu Komendy Głównej usta...
324,68324,"Według dziennika ""Washington Post"", władze w I...","Według dziennika ""Washington Post"", władze w I..."
325,68498,"Rabini z ortodoksyjnego, religijno-nacjonalist...","Rabini z ortodoksyjnego, religijno-nacjonalist..."


In [247]:
def preview_df(df, limit=10):
    df = df[["conllu_index", "correct", "incorrect"]].head(limit)

    for idx, conllu_index, corr, incorr in df.itertuples():
        print("#### " + str(conllu_index))
        print(corr)
        print(incorr)

subj_verb_number_pp_attractor_df.to_csv("output_old/subj_verb_number_pp_attractor_df.csv", index=False)
preview_df(subj_verb_number_pp_attractor_df, limit=20)

#### 1377
Nie wszystkie tomiki Broniewskiego sprzed roku 1939 dochowały się na moich półkach: Trzy salwy, Troska i pieśń, Krzyk ostateczny.
Nie wszystkie tomiki Broniewskiego sprzed roku 1939 dochował się na moich półkach: Trzy salwy, Troska i pieśń, Krzyk ostateczny.
#### 2477
Krążki VideoCD oprócz ścieżki audio/wideo mogą zawierać m.in. menu, spis rozdziałów (podobnie jak w filmach na DVD), napisy oraz dodatkowe ścieżki dźwiękowe (stereo lub mono).
Krążki VideoCD oprócz ścieżki audio/wideo może zawierać m.in. menu, spis rozdziałów (podobnie jak w filmach na DVD), napisy oraz dodatkowe ścieżki dźwiękowe (stereo lub mono).
#### 2671
Towarzysze z Moskwy przeprowadzali w Polsce szkolenia dla działaczy SdRP o zasadach działania partii w sferze gospodarczej - "pozwalających zabezpieczyć się przed próbami wywłaszczenia".
Towarzysze z Moskwy przeprowadzał w Polsce szkolenia dla działaczy SdRP o zasadach działania partii w sferze gospodarczej - "pozwalających zabezpieczyć się przed próbami wy

In [237]:


change_number(deepcopy(sentences[3240]).to_tree().token)
# deepcopy(sentences[3240]).to_tree().token

{'id': 7,
 'form': 'trafiają',
 'lemma': 'trafiać',
 'upos': 'VERB',
 'xpos': 'fin:pl:ter:imperf',
 'feats': {'Aspect': 'Imp',
  'Mood': 'Ind',
  'Number': 'Sing',
  'Person': '3',
  'Tense': 'Pres',
  'VerbForm': 'Fin',
  'Voice': 'Act'},
 'head': 0,
 'deprel': 'root',
 'deps': [('root', 0)],
 'misc': None}

In [235]:


get_form("trafiać", "fin:pl:ter:imperf")

'trafiają'

Problem - niespojność polimorf z UD, np:

* w UD mamy 'praet:pl:m1:imperf' a w polimorf nie ma
* w UD mamy 'ppron3:sg:nom:m3:ter:akc:npraep', a w polimorf nie ma  ( 'ppron3:sg:nom:m1.m2.m3:ter:_:_' ?)  (ona/on)
